# AML Capstone Project — EDA & Data Ingestion Walkthrough

This notebook demonstrates the step-by-step data ingestion, schema validation, deduplication, and exploratory data analysis (EDA) for the UCI Heart Disease multi-center dataset.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from data_loader import load_dataset, validate_dataset, deduplicate_records, clean_features, FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES

# Load raw dataset
raw_df = load_dataset()
print(f"Raw dataset dimensions: {raw_df.shape}")
raw_df.head()

## 2. Schema Validation & Deduplication

In [ ]:
# Validate raw dataset structure
validate_dataset(raw_df)
print("✓ Schema validation passed.")

# Deduplicate records ignoring arbitrary patient ID
df_dedup = deduplicate_records(raw_df)
duplicates_removed = len(raw_df) - len(df_dedup)
print(f"Duplicates detected and removed: {duplicates_removed}")
print(f"Final analysis cohort size: {len(df_dedup)}")

## 3. Clinical Cleaning & Zero-as-Missing Handling

In [ ]:
# Apply physiological cleaning (0 trestbps/chol -> NaN)
df_cleaned = clean_features(df_dedup)

missing_summary = pd.DataFrame({
    "Missing Count": df_cleaned.isna().sum(),
    "Missing (%)": (df_cleaned.isna().sum() / len(df_cleaned)) * 100
})
missing_summary.sort_values(by="Missing Count", ascending=False)

## 4. Target Distribution & Multi-Center Prevalence

In [ ]:
# Binary target: num > 0
positive = (df_dedup["num"] > 0).astype(int)
print("Binary Target Counts:")
print(positive.value_counts(normalize=True).rename({0: "Negative (No Disease)", 1: "Positive (Heart Disease)"}).mul(100).round(1).astype(str) + "%")

# Breakdown across collection sites
site_breakdown = df_dedup.groupby("dataset").agg(
    Total_Patients=("num", "count"),
    Positive_Count=("num", lambda s: (s > 0).sum()),
    Prevalence_Pct=("num", lambda s: (s > 0).mean() * 100)
).round(1)
site_breakdown

## 5. Descriptive Statistics & Correlation Matrix

In [ ]:
# Continuous feature statistics
print("Numeric Predictors Descriptives:")
display = df_cleaned[NUMERIC_FEATURES].describe().round(2)
display

In [ ]:
# Pearson correlation matrix
corr = df_cleaned[NUMERIC_FEATURES].corr().round(3)
corr